# DOGE Combined Analysis
Joins employment baselines to separation rates to answer: do pre-DOGE workforce characteristics predict which agencies were hit hardest?

In [ ]:
from pathlib import Path
import sys
import pickle

sys.path.append(str(Path.cwd().parent))
from scripts.data_loader import load_r2_data
import pandas as pd
import numpy as np

## Load Data

In [ ]:
# Load employment from cache (built in employment.ipynb)
CACHE = Path("employment.pkl")
with open(CACHE, "rb") as f:
    e_df = pickle.load(f)
e_df['ym'] = e_df['year'].astype(str) + '-' + e_df['month'].astype(str).str.zfill(2)
print(f"Employment: {e_df.shape[0]:,} rows")

In [ ]:
# Load separations
s_df = load_r2_data("separations")

INVOLUNTARY = {"SH", "SJ"}
s_df["sep_class"] = s_df["separation_category_code"].apply(
    lambda c: "involuntary" if c in INVOLUNTARY else ("voluntary" if c in {"SC","SD","SE","SG","SA","SB"} else "other")
)
s_df['ym'] = s_df['year'].astype(str) + '-' + s_df['month'].astype(str).str.zfill(2)
print(f"Separations: {s_df.shape[0]:,} rows")

## Build Pre-DOGE Agency Profile
Use Jan–Jun 2025 as the baseline. Compute % young, % non-tenured, % temp per agency.
Limit to agencies with ≥500 avg headcount to filter out noise from tiny offices.

In [ ]:
pre_doge = e_df[(e_df['year'] == 2025) & (e_df['month'] <= 6)].copy()
pre_doge_total = pre_doge.groupby('agency_subelement_code')['count'].sum().rename('total')

# Average monthly headcount
pre_doge_size = (
    pre_doge.groupby(['agency_subelement_code', 'agency_subelement', 'ym'])['count']
    .sum()
    .groupby(['agency_subelement_code', 'agency_subelement'])
    .mean().round(0).reset_index()
    .rename(columns={'count': 'avg_headcount_pre_doge'})
)

# % young (20-29)
young = (
    pre_doge[pre_doge['age_bracket'].isin(['20-24', '25-29'])]
    .groupby('agency_subelement_code')['count'].sum().rename('young_count')
)
pct_young = (young / pre_doge_total * 100).round(1).reset_index()
pct_young.columns = ['agency_subelement_code', 'pct_young']

# % non-tenured (probationary / term)
non_tenured = (
    pre_doge[pre_doge['tenure_code'].isin(['2', '3', 2.0, 3.0])]
    .groupby('agency_subelement_code')['count'].sum().rename('non_tenured_count')
)
pct_non_tenured = (non_tenured / pre_doge_total * 100).round(1).reset_index()
pct_non_tenured.columns = ['agency_subelement_code', 'pct_non_tenured']

# % temporary appointments
temp_codes = ['20', '30', '38', '50', '55']
temp_appt = (
    pre_doge[pre_doge['appointment_type_code'].isin(temp_codes)]
    .groupby('agency_subelement_code')['count'].sum().rename('temp_count')
)
pct_temp = (temp_appt / pre_doge_total * 100).round(1).reset_index()
pct_temp.columns = ['agency_subelement_code', 'pct_temp']

agency_profile = (
    pre_doge_size
    .merge(pct_young, on='agency_subelement_code', how='left')
    .merge(pct_non_tenured, on='agency_subelement_code', how='left')
    .merge(pct_temp, on='agency_subelement_code', how='left')
    .fillna(0)
)
agency_profile = agency_profile[agency_profile['avg_headcount_pre_doge'] >= 500]
print(f"Agencies in profile: {len(agency_profile)}")
agency_profile.sort_values('avg_headcount_pre_doge', ascending=False).head(10)

## Compute DOGE-Period Involuntary Separation Rates
Jul 2025 – Feb 2026 = the DOGE enforcement window. Compute avg and peak involuntary rate per agency.

In [ ]:
emp_by_agency_month = (
    e_df.groupby(['ym', 'agency_subelement_code'])['count']
    .sum().reset_index().rename(columns={'count': 'headcount'})
)

inv_by_agency_month = (
    s_df[s_df['sep_class'] == 'involuntary']
    .groupby(['ym', 'agency_subelement_code'])['count']
    .sum().reset_index().rename(columns={'count': 'inv_sep_count'})
)

rates = emp_by_agency_month.merge(inv_by_agency_month, on=['ym', 'agency_subelement_code'], how='left')
rates['inv_sep_count'] = rates['inv_sep_count'].fillna(0)
rates['inv_sep_rate_pct'] = (rates['inv_sep_count'] / rates['headcount'] * 100).round(3)

doge_rates = rates[rates['ym'] >= '2025-07']

agency_doge = (
    doge_rates.groupby('agency_subelement_code')
    .agg(
        avg_inv_sep_rate=('inv_sep_rate_pct', 'mean'),
        max_inv_sep_rate=('inv_sep_rate_pct', 'max'),
        total_inv_seps=('inv_sep_count', 'sum'),
    )
    .reset_index()
)

print(f"Agencies with DOGE-period data: {len(agency_doge)}")
agency_doge.sort_values('avg_inv_sep_rate', ascending=False).head(10)

## Regression: Do Pre-DOGE Characteristics Predict Separation Rate?
OLS: avg involuntary separation rate (Jul 2025–Feb 2026) ~ pct_young + pct_non_tenured + pct_temp + log(headcount)

In [ ]:
import statsmodels.formula.api as smf

reg_df = agency_profile.merge(agency_doge, on='agency_subelement_code', how='inner')
reg_df['log_headcount'] = np.log(reg_df['avg_headcount_pre_doge'])

print(f"Regression dataset: {len(reg_df)} agencies\n")

model = smf.ols(
    'avg_inv_sep_rate ~ pct_young + pct_non_tenured + pct_temp + log_headcount',
    data=reg_df
).fit()
print(model.summary())

In [ ]:
# Coefficients with confidence intervals — easier to read than the full summary
coef = pd.DataFrame({
    'coef': model.params,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
    'p_value': model.pvalues
}).round(4)
coef

## Workforce Collapse: High Separations + Large Headcount Loss
Agencies that lost ≥20% headcount AND had ≥100 total involuntary separations during the DOGE period.

In [ ]:
post_doge_size = (
    e_df[e_df['year'] == 2026]
    .groupby(['agency_subelement_code', 'agency_subelement', 'ym'])['count']
    .sum()
    .groupby(['agency_subelement_code', 'agency_subelement'])
    .mean().round(0).reset_index()
    .rename(columns={'count': 'avg_headcount_2026'})
)

collapse = pre_doge_size.merge(
    post_doge_size[['agency_subelement_code', 'avg_headcount_2026']],
    on='agency_subelement_code', how='inner'
)
collapse['pct_change'] = (
    (collapse['avg_headcount_2026'] - collapse['avg_headcount_pre_doge'])
    / collapse['avg_headcount_pre_doge'] * 100
).round(1)
collapse = collapse.merge(
    agency_doge[['agency_subelement_code', 'total_inv_seps', 'avg_inv_sep_rate', 'max_inv_sep_rate']],
    on='agency_subelement_code', how='left'
)
collapse[['total_inv_seps', 'avg_inv_sep_rate', 'max_inv_sep_rate']] = collapse[['total_inv_seps', 'avg_inv_sep_rate', 'max_inv_sep_rate']].fillna(0)

collapse_agencies = collapse[
    (collapse['pct_change'] <= -20) &
    (collapse['total_inv_seps'] >= 100) &
    (collapse['avg_headcount_pre_doge'] >= 500)
].sort_values('pct_change')

print(f"Agencies in workforce collapse: {len(collapse_agencies)}")
collapse_agencies[['agency_subelement', 'avg_headcount_pre_doge', 'avg_headcount_2026', 'pct_change', 'total_inv_seps', 'avg_inv_sep_rate']]

## Top Targeted Agencies: Full Profile
For the agencies with the highest involuntary separation rates, show their pre-DOGE characteristics side by side.

In [ ]:
# Merge profile + DOGE outcomes for top 25 agencies by avg involuntary rate
top25 = (
    reg_df.sort_values('avg_inv_sep_rate', ascending=False)
    .head(25)[[
        'agency_subelement',
        'avg_headcount_pre_doge',
        'pct_young',
        'pct_non_tenured',
        'pct_temp',
        'avg_inv_sep_rate',
        'max_inv_sep_rate',
        'total_inv_seps',
    ]]
)
top25